# Oedema Analysis Pipeline

--------

In [111]:
import fsl
from fsl.data.image import Image
from fsl.wrappers import flirt, bet, fast, fslmaths, applyxfm, fslstats
import numpy as np
import os

In [88]:
# raw files

path = '/Users/ecd25/Desktop/Fall_26/sub_18/'
T1 = os.path.join(path, 'anat', 'sub-18_ses-01_T1w.nii.gz')
FLAIR = os.path.join(path, 'anat', 'sub-18_ses-01_FLAIR.nii.gz')
B0 = os.path.join(path, 'dwi', 'data_b0.nii.gz')
TUM = os.path.join(path, 'anat', 'tumour_mask_ED_FLAIR.nii.gz')
inv_TUM = os.path.join(path, 'anat', 'sub-18_ses-01_inv_TUM.nii.gz')

In [119]:
# outputs

FLAIR_to_T1_mat = os.path.join(path, 'anat', 'sub-18_ses-01_FLAIR_to_T1.mat')
TUM_in_T1 = os.path.join(path, 'anat', 'sub-18_ses-01_TUM_in_T1.nii.gz')
TUM_thr = os.path.join(path, 'anat', 'sub-18_ses-01_TUM_thr.nii.gz')
TUM_bin = os.path.join(path, 'anat', 'sub-18_ses-01_TUM_bin.nii.gz')
T1_brain = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_brain.nii.gz')
T1_brain_mask = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_brain_mask.nii.gz')
T1_NT = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_NT.nii.gz')
FAST = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_NT_fast')

BF = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_NT_fast_bias.nii.gz')
T1_to_B0_mat = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_to_B0.mat')
BF_in_B0 = os.path.join(path, 'anat', 'sub-18_ses-01_BF_in_B0.nii.gz')
B0_bias_corrected = os.path.join(path, 'dwi', 'b0_bias_corrected.nii.gz')
WM = os.path.join(path, 'anat', 'sub-18_ses-01_T1w_NT_fast_pve_2.nii.gz')
WM_in_B0 = os.path.join(path, 'anat', 'sub-18_ses-01_WM_in_B0.nii.gz')
WM_thr = os.path.join(path, 'anat', 'sub-18_ses-01_WM_thr.nii.gz')
BF_in_B0_WM_sig = os.path.join(path, 'anat', 'sub-18_ses-01_BF_in_B0_WM_sig.nii.gz')
OEDEMA = os.path.join(path, 'dwi', 'OEDEMA.nii.gz')

In [ ]:
def struc_preproc(FLAIR: In("input/FLAIR"), T1: In("input/T1"), TUM: In("input/TUM"), T1_brain: Out("output/T1_brain"), T1_NT: Out("output/T1_NT"), TUM_bin: Out):
    """
    This stage receives a tumour mask (segmented in FLAIR), registers it to T1w space, and thresholds and binarizes it
    """
    # Log the script execution details
    print(f"Processing {TUM}")

    print("Registering FLAIR to T1")

    flirt(FLAIR, T1, omat=FLAIR_to_T1_mat)

    print("Applying transformation to TUM")

    applyxfm(TUM, T1, FLAIR_to_T1_mat, out=TUM_in_T1)

    print("Thresholding and binarizing TUM mask")

    fslmaths(TUM_in_T1).thr(0.8).bin().run(TUM_bin)

    print("Performing brain extraction on T1")

    bet(T1, T1_brain, mask=T1_brain_mask)

    print("Removing tumour from T1 brain mask")

    fslmaths(T1_brain_mask).sub(TUM_in_T1).thr(0.7).bin().run(inv_TUM)
    fslmaths(T1_brain).mas(inv_TUM).run(T1_NT)

    print("Preprocessing complete.")

def tbss_2_reg_fnirt(cleaned_FA: In, target_FA: In, FA_mask: In, warp: Out, affine: Out, warp_basename: Ref, config='legacy'):
    """
    Stage 2 of tbss_pipe, registering all the FA images to the target image.
    This function runs fnirt with a special fnirt config file that is provided by the user.
    """
    if config == 'legacy':
        config = "FA_2_FMRIB58_1mm"

    flirt(ref=target_FA, src=cleaned_FA, inweight=FA_mask, omat=affine, stderr=True)
    fnirt(src=cleaned_FA, ref=target_FA, cout=warp, config=config, aff=affine, logout=warp_basename + ".log", stderr=True)






In [ ]:
flirt(FLAIR, T1, omat=FLAIR_to_T1_mat)

{}

In [60]:
applyxfm(TUM, T1, FLAIR_to_T1_mat, out=TUM_in_T1)

{}

In [62]:
fslmaths(TUM_in_T1).thr(0.8).bin().run(TUM_bin)

{}

In [64]:
bet(T1, T1_brain, mask=T1_brain_mask)

{}

In [70]:
fslmaths(T1_brain_mask).sub(TUM_in_T1).thr(0.7).bin().run(inv_TUM)
fslmaths(T1_brain).mas(inv_TUM).run(T1_NT)

{}

In [ ]:
def oedema_pipeline:
    fast(T1_NT, out=FAST, b=True)
    flirt(T1, B0, omat=T1_to_B0_mat)
    applyxfm(BF, B0, T1_to_B0_mat, out=BF_in_B0)
    fslmaths(B0).div(BF_in_B0).run(B0_bias_corrected)
    applyxfm(WM, B0, T1_to_B0_mat, out=WM_in_B0)
    fslmaths(WM_in_B0).thr(0.8).bin().run(WM_thr)
    WM_sig = fslstats(B0_bias_corrected).k(WM_thr).M.run()
    print("WM_sig:", WM_sig)
    fslmaths(BF_in_B0).mul(WM_sig).run(BF_in_B0_WM_sig)
    fslmaths(B0).div(BF_in_B0_WM_sig).run(OEDEMA)

In [87]:
fast(T1_NT, out=FAST, b=True)

{}

In [95]:
flirt(T1, B0, omat=T1_to_B0_mat)

{}

In [96]:
applyxfm(BF, B0, T1_to_B0_mat, out=BF_in_B0)

{}

In [97]:
fslmaths(B0).div(BF_in_B0).run(B0_bias_corrected)

{}

In [99]:
applyxfm(WM, B0, T1_to_B0_mat, out=WM_in_B0)

{}

In [103]:
fslmaths(WM_in_B0).thr(0.8).bin().run(WM_thr)

{}

In [114]:
WM_sig = fslstats(B0_bias_corrected).k(WM_thr).M.run()

In [115]:
print("WM_sig:", WM_sig)

WM_sig: 571.016427


In [118]:
fslmaths(BF_in_B0).mul(WM_sig).run(BF_in_B0_WM_sig)

{}

In [120]:
fslmaths(B0).div(BF_in_B0_WM_sig).run(OEDEMA)

{}

In [ ]:
input
	raw
		sub-{subject}
			anat
				sub-{subject}_ses-{ses}_T1w.nii.gz
				sub-{subject}_ses-{ses}_FLAIR.nii.gz
			masks
				sub-{subject}_TUM.nii.gz	
			dwi
				sub-{subject}_ses-{ses}_dwi.nii.gz
				sub-{subject}_ses-{ses}_dwi.bvals
				sub-{subject}_ses-{ses}_dwi.bvecs
output
	derivatives
		sub-{subject}
			anat
				sub-{subject}_ses-{ses}_T1w_brain.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_mask.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_bias.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_mixeltype.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_pveseg.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_pve_2.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_pve_1.nii.gz
				sub-{subject}_ses-{ses}_T1w_brain_NT_pve_0.nii.gz
			masks   
        	    sub-{subject}_TUM_inv.nii.gz
				sub-{subject}_TUM_in_T1w.nii.gz
				sub-{subject}_TUM_in_T1w_bin.nii.gz
				sub-{subject}_TUM_in_diff.nii.gz
				sub-{subject}_TUM_in_diff_thr_bin.nii.gz
			dwi
                sub-{subject}_ses-{ses}_dwi_av_b0.nii.gz
				sub-{subject}_ses-{ses}_dwi_bias_in_diff.nii.gz
				sub-{subject}_ses-{ses}_dwi_b0_corrected.nii.gz
				sub-{subject}_ses-{ses}_dwi_WM_in_diff.nii.gz
				sub-{subject}_ses-{ses}_dwi_WM_thr.nii.gz
				sub-{subject}_ses-{ses}_dwi_bias_in_b0_WM.nii.gz
				sub-{subject}_ses-{ses}_dwi_OEDEMA.nii.gz
				sub-{subject}_ses-{ses}_dwi_signal.txt
			xfms
				sub-{subject}_ses-{ses}_FLAIR_to_T1w.mat
				sub-{subject}_ses-{ses}_T1w_to_diff.mat